In [26]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import camelot
import pandas
import numpy as np

c:\Users\DELL\coe\.venv\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Have the User Put in Pages to Process

In [4]:
pgs = "all"

In [6]:
tables = camelot.read_pdf("cg_files/Client Associates.pdf", pages=pgs,flavor='stream')

In [9]:
print(tables[5])

<Table shape=(45, 14)>


In [18]:
df_main = pandas.DataFrame()

for table in tables:
    df_main = pandas.concat([df_main, table.df], ignore_index=True)

Julius Bar Logic

In [5]:
key_words = ['Direct Equity','Sell Date']

In [6]:
r1 = df_main[0].value_counts()['Sell Date']

In [7]:
for x in range(0,r1):
    start_label_index = df_main.index[df_main[0] == key_words[0]][0]
    end_label_index = df_main.index[df_main[0] == key_words[1]][0]
    start_pos = df_main.index.get_loc(start_label_index)
    end_pos = df_main.index.get_loc(end_label_index)

    end_pos = end_pos+3
    indices_to_drop = df_main.iloc[start_pos:end_pos].index

    df_main.drop(indices_to_drop, inplace=True)

In [8]:
words = ["Vimuras Family Private Trust", "\\*Refer Disclaimer at the end of the report.","Equity","Taxable Gain & Loss Statement"]
pattern = "|".join(words)

Check each row if it has the above if so then delete the row, proboly need to loop

In [9]:
df_main = df_main[~df_main[0].str.contains(pattern, na=False, case=False)]
df_main = df_main[~df_main[5].str.contains(pattern, na=False, case=False)]

In [10]:
df_main.reset_index(inplace=True,drop=True)

In [11]:
def is_comp(val):
    if not isinstance(val, str):
        return False

    # 2. Strip whitespace from the string. This is the key change.
    #    "   Company A   " -> "Company A"
    #    "       "         -> ""
    cleaned_val = val.strip()

    # 3. Check if the cleaned string is empty. If it is, it's not a company name.
    #    This now correctly handles strings that were originally just whitespace.
    if not cleaned_val:
        return False

    # 4. Now that we know we have a non-empty, cleaned string, apply the final rules.
    if cleaned_val.lower() == 'total' or cleaned_val[0].isdigit():
        return False

    # 5. If it passed all the checks, it's a company name.
    return True


In [12]:
mask1 = df_main[0].apply(is_comp)
mask2 = df_main[1].apply(is_comp)


In [13]:
mask1.value_counts()

0
False    2492
True      201
Name: count, dtype: int64

In [14]:
combine_mask = mask1 | mask2

In [15]:
company_names = pandas.Series(np.where(mask1, df_main[0], df_main[1]))

In [16]:
company_col = company_names.where(combine_mask).ffill()

In [17]:
df_main = df_main[~combine_mask]

In [18]:
df_main.insert(0, "Company", company_col)

In [19]:
df_main.columns = ["Company", "Sell Date", "Quantity", "Sell Rate", "Total Sale Value", "Purchase Date", "Purchase Rate","Actual Cost", "FMV as on 31-01-2018/Indexed Rate", "Applicable Rate", "Effective Cost", "Days Held", "Short Term", "Long Term", "Effective LT"]

In [26]:
df_main.to_excel("JBCAP_PRO.xlsx")

In [23]:
df_c = pandas.read_csv("JBCAP PRO")
df_c = df_c.drop('Unnamed: 0', axis=1)

In [25]:
df_c.to_excel("JB Exl.xlsx")

HDFC Logic

In [30]:
tables = camelot.read_pdf("cg_files/cams.pdf", pages="2-3",flavor='lattice')

In [50]:
df_main = pandas.DataFrame()

for table in tables:
    df_main = pandas.concat([df_main, table.df], ignore_index=True)

In [51]:
cams_key = ["Scheme Name", "TOTAL"]

In [52]:
start_label_index = df_main.index[df_main[0] == cams_key[0]][0]
end_label_index = df_main.index[df_main[0] == cams_key[1]][0]

start_pos = df_main.index.get_loc(start_label_index)
end_pos = df_main.index.get_loc(end_label_index)

In [53]:
df_main = df_main.iloc[start_pos:end_pos+1,:]

In [54]:
mask1 = df_main[5].apply(is_comp)

In [55]:
df_main = df_main[~mask1]

In [56]:
df_main.columns = ["Scheme Name", "Total Count", "Total Amount", "Total Cost", "Indexed Cost", "Grandfathered Value, Market Value as on 31/01/2018", "Short Term","LongTerm with Indexation", "LongTerm without Indexation", "TDS Amount"]

ICICI Prudential

In [2]:
import camelot
import pandas
import numpy as np

In [3]:
tables = camelot.read_pdf("cg_files/current.pdf", pages="1-16",flavor='stream')

In [4]:
df_main = pandas.DataFrame()

for table in tables:
    df_main = pandas.concat([df_main, table.df], ignore_index=True)

In [14]:
icici_key = ["STATEMENT OF CAPITAL GAIN/LOSS", "Listed Shares/Equity Mutual Funds (STT paid on Sale)"]

In [21]:
r1 = df_main[0].value_counts()[icici_key[0]]

In [22]:
for x in range(0,r1):
    start_label_index = df_main.index[df_main[0] == icici_key[0]][0]
    end_label_index = df_main.index[df_main[0] == icici_key[1]][0]
    start_pos = df_main.index.get_loc(start_label_index)
    end_pos = df_main.index.get_loc(end_label_index)

    end_pos = end_pos+3
    indices_to_drop = df_main.iloc[start_pos:end_pos].index

    df_main.drop(indices_to_drop, inplace=True)

In [25]:
df_main.to_excel("Sanjay Prasad2.xlsx")

In [24]:
df_main.columns = ["Company", "Sale Date", "Quantity", "Sale Rate", "Sale Amount", "Purchase Date", "Purchase Rate","Price on 31-Jan-18(M)", "Purchase Amount", "Effective Cost", "Days Held", "ST", "LT", "Effective Gain-LT"]

In [9]:
import camelot
import pandas
pgs = "all"

Client Associates

In [10]:
tables = camelot.read_pdf("cg_files/Client Associates.pdf", pages=pgs,flavor='stream')

In [11]:
df_main = pandas.DataFrame()

for table in tables:
    df_main = pandas.concat([df_main, table.df], ignore_index=True)

In [12]:
print(tables[5])

<Table shape=(45, 14)>


In [13]:
c_a_key = ["Security", "Listed Shares/Equity Mutual Funds (STT paid on Sale)"]

In [14]:
r1 = df_main[0].value_counts()[c_a_key[0]]

In [15]:
for x in range(0,r1):
    start_label_index = df_main.index[df_main[0] == c_a_key[0]][0]
    end_label_index = df_main.index[df_main[0] == c_a_key[1]][0]
    start_pos = df_main.index.get_loc(start_label_index)
    end_pos = df_main.index.get_loc(end_label_index)

    end_pos = end_pos+3
    indices_to_drop = df_main.iloc[start_pos:end_pos].index

    df_main.drop(indices_to_drop, inplace=True)

In [16]:
words = ["CAPITAL GAIN", "This is for informational purposes only. Please verify with your tax consultant.", "Account"]
pattern = "|".join(words)

In [17]:
df_main = df_main[~df_main[0].str.contains(pattern, na=False, case=False)]

In [23]:
df_main = df_main[~df_main[3].str.contains("Sale", na=False, case=False)]
df_main = df_main[~df_main[2].str.contains("Sale", na=False, case=False)]

Upstox

In [24]:
tables = camelot.read_pdf("cg_files/upstox.pdf", pages=pgs,flavor='stream')

In [25]:
df_main = pandas.DataFrame()

for table in tables:
    df_main = pandas.concat([df_main, table.df], ignore_index=True)

In [26]:
df_main = df_main[~df_main[0].str.contains("Date", na=False,case=False)]

In [1]:
%pip install xlsxwriter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip
